# C1 · 階層模型與「部分匯聚」的魔法

> **核心問題**：某個郡只量了 2 棟房子，平均值很不可靠。**我該相信它自己的平均，還是全國平均？**
> 貝葉斯的答案：兩個都不要，要一個**按可靠度加權**的中間值。

資料：Radon（C-D1），919 戶、85 郡、樣本數 1–116。核心程式在 [`../src/`](../src)。

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, arviz as az
import data, models, shrinkage as shr
rd = data.load_radon()
print(f'{rd.N} 戶、{rd.J} 郡；郡樣本數 {rd.n_j.min()}–{rd.n_j.max()}（中位數 {int(np.median(rd.n_j))}）')
id_np = models.fit_no_pooling(rd.cidx, rd.floor, rd.y, rd.J)
id_cp = models.fit_complete_pooling(rd.cidx, rd.floor, rd.y, rd.J)
id_h  = models.fit_hierarchical(rd.cidx, rd.floor, rd.y, rd.J)
print('階層模型 divergences =', int(id_h.sample_stats['diverging'].sum()),
      '| max r_hat =', round(float(az.summary(id_h, var_names=['mu_a','sigma_a','b'])['r_hat'].max()),3))

Initializing NUTS using jitter+adapt_diag...


919 戶、85 郡；郡樣本數 1–116（中位數 5）


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [a, b, sigma]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1 seconds.


...

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [mu_a, sigma_a, a_raw, b, sigma]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 2 seconds.


階層模型 divergences = 0 | max r_hat = 1.0


## 1 · 兩個極端 + 部分匯聚

| 模型 | 做法 | 問題 |
|---|---|---|
| **no pooling** | 每郡各算各的 | 小樣本郡估計極不穩 |
| **complete pooling** | 全部混一起算一個 | 完全忽略郡的差異 |
| **hierarchical** | $a_j \sim \mathcal{N}(\mu_a, \sigma_a)$，超先驗學出母體分佈 | **部分匯聚**——恰到好處 |

階層模型用**非中心化參數化**避免 funnel（見 [`02_funnel_noncentered.ipynb`](02_funnel_noncentered.ipynb)）。

### 三種做法並排（示例郡）
![三種做法](../figures/01_three_approaches.png)

小郡（MAHNOMEN n=1、COOK n=2）的 no-pooling 估計誤差棒巨大，被部分匯聚**收窄並拉向全國平均**；
大郡（ST LOUIS n=116）三者幾乎重合——**資料夠多時，部分匯聚自動放手。**

## 2 · 收縮圖（本專案靈魂）

每個郡有兩個估計：只用自己資料的 no-pooling、以及借力全體的 partial pooling。收縮＝後者把前者拉向全國平均。

In [2]:
a_np = models.county_intercepts(id_np, rd.J)
a_pp = models.county_intercepts(id_h,  rd.J)
a_cp = models.county_intercepts(id_cp, rd.J)[0]
omega = shr.empirical_weight(a_np, a_pp, a_cp)
mask = np.abs(a_np - a_cp) > 0.1
print(f'樣本數≤3 的郡：平均收縮權重 = {np.nanmean(omega[(rd.n_j<=3)&mask]):.2f}')
print(f'樣本數≥30 的郡：平均收縮權重 = {np.nanmean(omega[(rd.n_j>=30)&mask]):.2f}')

樣本數≤3 的郡：平均收縮權重 = 0.64
樣本數≥30 的郡：平均收縮權重 = 0.14


![收縮圖](../figures/02_shrinkage.png)

**左**：小 n 郡（左側）的 no-pooling（紅圈）散得很開，partial pooling（藍）被拉向 grand mean；大 n 郡（右側）兩者重合。
**右**：收縮權重 ω 隨 n 遞減（≤3 的郡 ω≈0.64 → ≥30 的郡 ω≈0.14）。

> **這其實是你學過的東西**：部分匯聚 $=$ 精確度加權平均。
> $a_j^{\text{partial}} \approx (1-\omega_j)\,a_j^{\text{own}} + \omega_j\,\mu_a$，其中 $\omega_j = \dfrac{\sigma^2}{\sigma^2 + n_j\sigma_a^2}$。
> 圖右的橘線就是這條理論曲線——郡自己的資料是「似然」、母體分佈是「先驗」，只是這次**先驗本身也從資料學出來**（empirical Bayes / 階層貝葉斯）。

## 3 · 郡與郡之間差異有多大？（σ_a）

$\sigma_a$ 是「郡間標準差」——**這正是完全匯聚模型無法回答的問題**（它假設 $\sigma_a=0$）。

In [3]:
sig_a = id_h.posterior['sigma_a'].values.reshape(-1)
lo, med, hi = np.percentile(sig_a, [2.5, 50, 97.5])
print(f'σ_a 後驗：中位數={med:.3f}，95% CI=[{lo:.3f}, {hi:.3f}] → 明確 >0')

σ_a 後驗：中位數=0.319，95% CI=[0.241, 0.417] → 明確 >0


![超參數](../figures/03_hyperparameters.png)

後驗質量整個遠離 0（灰虛線）——資料明確告訴我們「郡之間真的有差異」。這是階層模型獨有的答案。

## 4 · 預測驗證：階層模型真的比較好嗎？

留出測試集，比較三個模型的預測誤差（下方示範單次切分；[`run_all.py`](../src/run_all.py) 對 10 次切分取平均生成下圖）。

In [4]:
tr, te = data.train_test_split(rd, test_frac=0.25, seed=0)
for name, fit in [('no pooling', models.fit_no_pooling),
                  ('complete pooling', models.fit_complete_pooling),
                  ('hierarchical', models.fit_hierarchical)]:
    idata = fit(rd.cidx[tr], rd.floor[tr], rd.y[tr], rd.J, draws=500, tune=500, chains=2)
    pred = models.predict(idata, rd.cidx[te], rd.floor[te], rd.J)
    print(f'{name:<18} 測試 RMSE = {shr.rmse(pred, rd.y[te]):.3f}')

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [a, b, sigma]


Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 1 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


...

NUTS: [mu_a, sigma_a, a_raw, b, sigma]


Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 1 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


hierarchical       測試 RMSE = 0.763


![預測](../figures/04_predictive.png)

10 次切分平均：**hierarchical 0.765 < complete 0.791 < no-pooling 0.797**——階層模型整體最低，
而且它在**每個郡大小分桶都最好**，對 no-pooling 的優勢在**小郡最大**（0.873→0.803）、大郡趨近。

## 重點

1. 「該信自己還是全國平均」的答案是**按可靠度（精確度）加權**——這就是部分匯聚。
2. 收縮的拉力隨樣本數遞減，且**精確地**等於精確度加權平均。
3. $\sigma_a$ 的後驗回答「群組間差異多大」，完全匯聚問不出來。
4. 階層模型在預測上勝過兩個極端，尤其對小樣本群組。